# KLA Hackathon -- Image Restoration Pipeline

Everything from the `swinir-approach` branch in one runnable notebook:
calibration, dataset, both model architectures (Restormer and SwinIR),
the shared training engine, both training loops, and evaluation.

Run top to bottom. Each stage is a plain function you call yourself at the
bottom (Section 9) -- nothing auto-executes on import, since training
takes hours.

See `swinirapproach.md` in the repo for the write-up of results this
pipeline produced (12-epoch quick-check and full 50-epoch comparison
between the two architectures).

In [1]:
import os
import glob
import json
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Calibration stats

Computes p1/p99.5 ONCE over the training set's noisy-LR images (that's
where the out-of-range speckle spikes live) and saves them to
`calib_stats.json`. The same two numbers are reused to normalize LR, GT,
and test data everywhere else -- recomputing per-split would make every
image's "1.0" mean a different physical intensity.

In [2]:
def compute_global_percentiles(lr_dir, p_low=1.0, p_high=99.5, sample_cap=None):
    files = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))
    if not files:
        raise ValueError(f"No .npy files found in {lr_dir}")

    if sample_cap is not None and len(files) > sample_cap:
        rng = np.random.default_rng(42)
        files = list(rng.choice(files, size=sample_cap, replace=False))

    print(f"Computing global percentiles over {len(files)} LR files...")

    all_values = []
    for f in files:
        arr = np.load(f).astype(np.float32)
        all_values.append(arr.ravel())

    all_values = np.concatenate(all_values)
    p_lo = float(np.percentile(all_values, p_low))
    p_hi = float(np.percentile(all_values, p_high))

    if p_hi <= p_lo:
        raise ValueError(
            f"p{p_high}={p_hi} <= p{p_low}={p_lo} -- data looks degenerate "
            f"(near-constant?). Check the files in {lr_dir}."
        )

    print(f"p{p_low} = {p_lo:.6f}, p{p_high} = {p_hi:.6f}")
    print(f"raw min = {all_values.min():.6f}, raw max = {all_values.max():.6f}")
    frac_clipped_low = (all_values < p_lo).mean()
    frac_clipped_high = (all_values > p_hi).mean()
    print(f"fraction below p{p_low}: {frac_clipped_low:.4%}, "
          f"fraction above p{p_high}: {frac_clipped_high:.4%}")

    return p_lo, p_hi


def run_calibration(data_root, p_low=1.0, p_high=99.5):
    """Writes calib_stats.json into data_root. Run once before training."""
    lr_dir = os.path.join(data_root, "train", "NoisyLR")
    p_lo, p_hi = compute_global_percentiles(lr_dir, p_low=p_low, p_high=p_high)

    stats = {"p_low": p_lo, "p_high": p_hi, "p_low_pct": p_low, "p_high_pct": p_high}
    out_path = os.path.join(data_root, "calib_stats.json")
    with open(out_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"Saved calibration stats to {out_path}")
    return p_lo, p_hi

## 2. Dataset / DataLoaders

Folder layout assumed:
```
train/gt/*.npy       -- clean, 256x256
train/NoisyLR/*.npy  -- noisy, 128x128
test/*.npy           -- noisy, 128x128, no gt
```

In [3]:
def load_calib_stats(data_root):
    path = os.path.join(data_root, "calib_stats.json")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Run run_calibration(data_root) first -- "
            f"normalization must be computed once and reused everywhere, "
            f"not recomputed ad hoc."
        )
    with open(path) as f:
        stats = json.load(f)
    return stats["p_low"], stats["p_high"]


def normalize(arr, p_low, p_high):
    """Affine map: p_low -> 0, p_high -> 1. Values outside [p_low, p_high]
    (the speckle spikes) are clamped, not stretched -- this is the
    'robust percentile scaling' step."""
    arr = (arr - p_low) / (p_high - p_low)
    return np.clip(arr, 0.0, 1.0).astype(np.float32)


class SEMPairDataset(Dataset):
    """Paired GT / noisy-LR dataset for training, using shared calibration."""

    def __init__(self, gt_dir, lr_dir, p_low, p_high, scale_factor=2,
                 augment=True, strict_name_match=True):
        self.gt_dir = gt_dir
        self.lr_dir = lr_dir
        self.p_low = p_low
        self.p_high = p_high
        self.scale_factor = scale_factor
        self.augment = augment

        gt_files = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        lr_files = sorted(glob.glob(os.path.join(lr_dir, "*.npy")))

        if strict_name_match:
            gt_names = {os.path.basename(f) for f in gt_files}
            lr_names = {os.path.basename(f) for f in lr_files}
            common = gt_names & lr_names
            if len(common) != len(gt_names) or len(common) != len(lr_names):
                missing_in_lr = gt_names - lr_names
                missing_in_gt = lr_names - gt_names
                raise ValueError(
                    f"Filename mismatch between gt/ and NoisyLR/.\n"
                    f"In gt but not NoisyLR ({len(missing_in_lr)}): {list(missing_in_lr)[:5]}...\n"
                    f"In NoisyLR but not gt ({len(missing_in_gt)}): {list(missing_in_gt)[:5]}...\n"
                    f"Pass strict_name_match=False if names differ but are index-aligned."
                )
            self.pairs = [(os.path.join(gt_dir, n), os.path.join(lr_dir, n))
                          for n in sorted(common)]
        else:
            if len(gt_files) != len(lr_files):
                raise ValueError(
                    f"gt/ has {len(gt_files)} files, NoisyLR/ has {len(lr_files)} "
                    f"-- can't pair positionally."
                )
            self.pairs = list(zip(gt_files, lr_files))

        print(f"[SEMPairDataset] {len(self.pairs)} paired samples found.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        gt_path, lr_path = self.pairs[idx]
        gt = normalize(np.load(gt_path).astype(np.float32), self.p_low, self.p_high)
        lr = normalize(np.load(lr_path).astype(np.float32), self.p_low, self.p_high)

        expected_h = gt.shape[0] // self.scale_factor
        expected_w = gt.shape[1] // self.scale_factor
        if (lr.shape[0], lr.shape[1]) != (expected_h, expected_w):
            raise ValueError(
                f"Shape mismatch at {os.path.basename(lr_path)}: gt={gt.shape}, "
                f"lr={lr.shape}, expected lr={(expected_h, expected_w)} for "
                f"scale_factor={self.scale_factor}."
            )

        gt_t = torch.from_numpy(gt).unsqueeze(0)  # (1, H, W)
        lr_t = torch.from_numpy(lr).unsqueeze(0)  # (1, h, w)

        if self.augment:
            gt_t, lr_t = self._augment(gt_t, lr_t)

        return lr_t, gt_t

    @staticmethod
    def _augment(gt_t, lr_t):
        # Bit-perfect augmentation only -- torch.rot90 / torch.flip are exact
        # index permutations, unlike torchvision's interpolated rotate(),
        # which would blur precision float32 metrology data.
        if torch.rand(1).item() < 0.5:
            gt_t = torch.flip(gt_t, dims=[-1])
            lr_t = torch.flip(lr_t, dims=[-1])
        if torch.rand(1).item() < 0.5:
            gt_t = torch.flip(gt_t, dims=[-2])
            lr_t = torch.flip(lr_t, dims=[-2])
        k = torch.randint(0, 4, (1,)).item()
        if k:
            gt_t = torch.rot90(gt_t, k, dims=[-2, -1])
            lr_t = torch.rot90(lr_t, k, dims=[-2, -1])
        return gt_t, lr_t


class SEMTestDataset(Dataset):
    """Test set: noisy-LR only, no ground truth. Uses the SAME calibration
    stats as training -- never fit fresh stats on the test set."""

    def __init__(self, test_dir, p_low, p_high):
        self.p_low = p_low
        self.p_high = p_high
        self.files = sorted(glob.glob(os.path.join(test_dir, "*.npy")))
        if not self.files:
            raise ValueError(f"No .npy files found in {test_dir}")
        print(f"[SEMTestDataset] {len(self.files)} test samples found.")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        lr = normalize(np.load(path).astype(np.float32), self.p_low, self.p_high)
        lr_t = torch.from_numpy(lr).unsqueeze(0)
        return lr_t, os.path.basename(path)


class _NoAugWrapper(Dataset):
    """Wraps a Subset of SEMPairDataset and forces augment=False for reads
    that go through it, without permanently mutating the shared underlying
    dataset."""

    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        ds = self.subset.dataset
        real_idx = self.subset.indices[idx]
        was_augment = ds.augment
        ds.augment = False
        try:
            item = ds[real_idx]
        finally:
            ds.augment = was_augment
        return item


def build_dataloaders(data_root, scale_factor=2, batch_size=4, num_workers=8,
                       val_fraction=0.1, seed=42):
    p_low, p_high = load_calib_stats(data_root)

    full_train = SEMPairDataset(
        gt_dir=os.path.join(data_root, "train", "gt"),
        lr_dir=os.path.join(data_root, "train", "NoisyLR"),
        p_low=p_low, p_high=p_high,
        scale_factor=scale_factor,
        augment=True,
    )

    n_val = max(1, int(len(full_train) * val_fraction))
    n_train = len(full_train) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = torch.utils.data.random_split(
        full_train, [n_train, n_val], generator=generator
    )
    val_set = _NoAugWrapper(val_subset)

    test_set = SEMTestDataset(os.path.join(data_root, "test"), p_low, p_high)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True,
                               drop_last=True, prefetch_factor=2 if num_workers > 0 else None)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True,
                             prefetch_factor=2 if num_workers > 0 else None)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True,
                              prefetch_factor=2 if num_workers > 0 else None)

    return train_loader, val_loader, test_loader

## 3. Restormer model (`SemiRestoreNet_V2`) + shared loss / optimizer

Restormer-style channel attention (linear in pixel count, not quadratic)
plus a learned Fourier Unit for global frequency mixing. Global residual
learning on top of a bicubic upsample, with the last conv zero-initialized
so training starts at the bicubic baseline instead of noise.

`KLAMetrologyLoss` and `build_optimizer` here are architecture-agnostic
and reused by the SwinIR model below too.

In [4]:
class FourierUnit(nn.Module):
    """Global frequency-domain mixer. Learned, not a hand-crafted notch
    filter -- gives cheap global receptive field, still needs training
    signal to actually suppress periodic scan-line noise."""

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels * 2, channels * 2, 1)
        self.norm = nn.GroupNorm(1, channels * 2)

    def forward(self, x):
        B, C, H, W = x.shape
        fft = torch.fft.rfft2(x.float(), norm='ortho')  # fp32: not autocast-safe otherwise
        fft_real = torch.stack([fft.real, fft.imag], dim=1)
        fft_filt = self.norm(self.conv(fft_real.view(B, 2 * C, H, -1)))
        fft_filt = fft_filt.view(B, 2, C, H, -1)
        fft_final = torch.complex(fft_filt[:, 0], fft_filt[:, 1])
        return torch.fft.irfft2(fft_final, s=(H, W), norm='ortho').to(x.dtype)


class MetrologyRestormerBlock(nn.Module):
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.norm1 = nn.GroupNorm(1, channels)
        self.norm2 = nn.GroupNorm(1, channels)

        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.qkv_dw = nn.Conv2d(channels * 3, channels * 3, 3, padding=1, groups=channels * 3)
        self.project_out = nn.Conv2d(channels, channels, 1)
        self.spectral = FourierUnit(channels)

        self.gdfn_in = nn.Conv2d(channels, channels * 4, 1)
        self.gdfn_dw = nn.Conv2d(channels * 4, channels * 4, 3, padding=1, groups=channels * 4)
        self.gdfn_out = nn.Conv2d(channels * 2, channels, 1)

    def _attn_forward(self, x):
        b, c, h, w = x.shape
        q, k, v = self.qkv_dw(self.qkv(x)).chunk(3, dim=1)
        q = F.normalize(q.view(b, self.num_heads, -1, h * w), dim=-1)
        k = F.normalize(k.view(b, self.num_heads, -1, h * w), dim=-1)
        v = v.view(b, self.num_heads, -1, h * w)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        return self.project_out((attn.softmax(dim=-1) @ v).view(b, c, h, w))

    def forward(self, x):
        x_norm = self.norm1(x)
        x = x + self._attn_forward(x_norm) + self.spectral(x_norm)

        x1, x2 = self.gdfn_dw(self.gdfn_in(self.norm2(x))).chunk(2, dim=1)
        x = x + self.gdfn_out(F.gelu(x1) * x2)
        return x


class SemiRestoreNet_V2(nn.Module):
    def __init__(self, dim=64, num_blocks=2, scale_factor=2):
        super().__init__()
        self.scale_factor = scale_factor
        self.embed = nn.Conv2d(1, dim, 3, padding=1)

        self.encoder = nn.Sequential(*[MetrologyRestormerBlock(dim) for _ in range(num_blocks)])
        self.bottleneck = MetrologyRestormerBlock(dim)
        self.decoder = nn.Sequential(*[MetrologyRestormerBlock(dim) for _ in range(num_blocks)])

        if scale_factor > 1:
            self.upsampler = nn.Sequential(
                nn.Conv2d(dim, dim * (scale_factor ** 2), 3, padding=1),
                nn.PixelShuffle(scale_factor),
                nn.Conv2d(dim, 1, 3, padding=1),
            )
            last_conv = self.upsampler[-1]
        else:
            self.upsampler = nn.Conv2d(dim, 1, 3, padding=1)
            last_conv = self.upsampler

        # Zero-init the final conv so the residual ("out") starts at exactly
        # zero -- forward() at step 0 returns clamp(0 + base, 0, 1), i.e. the
        # bicubic baseline itself, a far better starting point than noise.
        nn.init.zeros_(last_conv.weight)
        nn.init.zeros_(last_conv.bias)

    def forward(self, x):
        # x is assumed already normalized to [0, 1] by the shared percentile
        # calibration above -- no transform applied here.
        feat = self.embed(x)
        res = self.encoder(feat)
        res = self.bottleneck(res)
        res = self.decoder(res + feat)
        out = self.upsampler(res)

        if self.scale_factor > 1:
            base = F.interpolate(x, scale_factor=self.scale_factor,
                                  mode='bicubic', align_corners=False)
        else:
            base = x

        return torch.clamp(out + base, 0.0, 1.0)


class KLAMetrologyLoss(nn.Module):
    """Charbonnier + edge (Sobel) loss, with optional frequency and SSIM
    terms. SSIM (1 - SSIM) targets the over-smoothing ("melting") that pure
    L1-style losses are documented to cause under high noise."""

    def __init__(self, edge_weight=0.5, freq_weight=0.0, ssim_weight=0.0, ms_ssim_weight=0.0):
        super().__init__()
        self.edge_weight = edge_weight
        self.freq_weight = freq_weight
        self.ssim_weight = ssim_weight
        self.ms_ssim_weight = ms_ssim_weight  # multi-scale SSIM term
        self.register_buffer('sobel_x', torch.tensor(
            [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3))
        self.register_buffer('sobel_y', torch.tensor(
            [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3))
        self.register_buffer('_ssim_window', self._make_gaussian_window(11, 1.5))
        # Wang/Simoncelli/Bovik 2003 per-scale weights, coarsest scale last.
        self.register_buffer('_ms_ssim_weights',
                              torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333]))

    @staticmethod
    def _make_gaussian_window(window_size, sigma):
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        window_2d = g.unsqueeze(1) @ g.unsqueeze(0)
        return window_2d.unsqueeze(0).unsqueeze(0)  # (1, 1, k, k)

    def charbonnier_loss(self, pred, target, eps=1e-6):
        return torch.mean(torch.sqrt((pred - target) ** 2 + eps))

    def edge_loss(self, pred, target):
        pred_f, target_f = pred.float(), target.float()
        p_edge = F.conv2d(pred_f, self.sobel_x, padding=1) ** 2 + \
                 F.conv2d(pred_f, self.sobel_y, padding=1) ** 2
        t_edge = F.conv2d(target_f, self.sobel_x, padding=1) ** 2 + \
                 F.conv2d(target_f, self.sobel_y, padding=1) ** 2
        return F.l1_loss(p_edge, t_edge)

    def frequency_loss(self, pred, target):
        pred_fft = torch.fft.rfft2(pred.float(), norm='ortho')
        gt_fft = torch.fft.rfft2(target.float(), norm='ortho')
        return F.l1_loss(torch.abs(pred_fft), torch.abs(gt_fft))

    def ssim_loss(self, pred, target, data_range=1.0):
        pred_f, target_f = pred.float(), target.float()
        window = self._ssim_window
        pad = window.shape[-1] // 2
        C1 = (0.01 * data_range) ** 2
        C2 = (0.03 * data_range) ** 2

        mu1 = F.conv2d(pred_f, window, padding=pad)
        mu2 = F.conv2d(target_f, window, padding=pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2

        sigma1_sq = F.conv2d(pred_f * pred_f, window, padding=pad) - mu1_sq
        sigma2_sq = F.conv2d(target_f * target_f, window, padding=pad) - mu2_sq
        sigma12 = F.conv2d(pred_f * target_f, window, padding=pad) - mu1_mu2

        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
                   ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        return 1.0 - ssim_map.mean()

    def _ssim_and_cs(self, pred, target, data_range=1.0):
        """Returns (full SSIM incl. luminance, contrast*structure only --
        the piece MS-SSIM reuses at every scale except the coarsest)."""
        window = self._ssim_window
        pad = window.shape[-1] // 2
        C1 = (0.01 * data_range) ** 2
        C2 = (0.03 * data_range) ** 2

        mu1 = F.conv2d(pred, window, padding=pad)
        mu2 = F.conv2d(target, window, padding=pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2

        sigma1_sq = F.conv2d(pred * pred, window, padding=pad) - mu1_sq
        sigma2_sq = F.conv2d(target * target, window, padding=pad) - mu2_sq
        sigma12 = F.conv2d(pred * target, window, padding=pad) - mu1_mu2

        cs_map = (2 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2)
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
                   ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean(), cs_map.mean()

    def ms_ssim(self, pred, target, data_range=1.0, eps=1e-6):
        """Multi-scale SSIM (Wang et al. 2003): contrast*structure at
        successively 2x-downsampled scales, geometric-mean-combined with
        full SSIM (incl. luminance) at the coarsest scale only. More
        robust than single-scale SSIM for content with structure at
        multiple frequencies (fine texture + coarse shape) at once.
        Needs input large enough for len(weights)-1 halvings to stay
        >= the 11x11 window -- fine at this pipeline's fixed 256x256 /
        128x128 resolutions, not general-purpose for arbitrary sizes.
        """
        pred_f, target_f = pred.float(), target.float()
        weights = self._ms_ssim_weights
        levels = weights.shape[0]

        cs_vals = []
        ssim_val = None
        x, y = pred_f, target_f
        for i in range(levels):
            s, cs = self._ssim_and_cs(x, y, data_range=data_range)
            cs_vals.append(cs.clamp(min=eps))
            if i == levels - 1:
                ssim_val = s.clamp(min=eps)
            else:
                x = F.avg_pool2d(x, kernel_size=2)
                y = F.avg_pool2d(y, kernel_size=2)

        cs_stack = torch.stack(cs_vals)
        return ssim_val ** weights[-1] * torch.prod(cs_stack[:-1] ** weights[:-1])

    def ms_ssim_loss(self, pred, target, data_range=1.0):
        return 1.0 - self.ms_ssim(pred, target, data_range=data_range)

    def forward(self, pred, target):
        l_char = self.charbonnier_loss(pred, target)
        l_edge = self.edge_loss(pred, target)
        loss = l_char + self.edge_weight * l_edge
        if self.freq_weight > 0:
            loss = loss + self.freq_weight * self.frequency_loss(pred, target)
        if self.ssim_weight > 0:
            loss = loss + self.ssim_weight * self.ssim_loss(pred, target)
        if self.ms_ssim_weight > 0:
            loss = loss + self.ms_ssim_weight * self.ms_ssim_loss(pred, target)
        return loss


def build_optimizer(model, lr=2e-4, weight_decay=1e-4):
    """Standard practice: exclude 1-D params (norms, biases, temperature)
    from weight decay."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or 'norm' in n or 'temperature' in n:
            no_decay.append(p)
        else:
            decay.append(p)
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': weight_decay},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=lr)

## 4. SwinIR model

Shifted-window self-attention (RSTB blocks) instead of Restormer's channel
attention -- a genuinely different inductive bias on the same
data/loss/training pipeline. Same global-residual + zero-init-last-conv
trick as the Restormer model above.

Note: two `transpose(1, 2).view(...)` chains need `.contiguous()` before
the `.view()` -- PyTorch refuses `.view()` on a non-contiguous tensor, and
`transpose()` always produces one. Already fixed below.

In [5]:
def to_2tuple(x):
    return (x, x) if isinstance(x, int) else tuple(x)


def window_partition(x, window_size):
    """(B, H, W, C) -> (num_windows*B, window_size, window_size, C)"""
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)


def window_reverse(windows, window_size, H, W):
    """(num_windows*B, window_size, window_size, C) -> (B, H, W, C)"""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.window_size = window_size  # (Wh, Ww)
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))

        coords = torch.stack(torch.meshgrid(
            torch.arange(window_size[0]), torch.arange(window_size[1]), indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size[0] - 1
        relative_coords[:, :, 1] += window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * window_size[1] - 1
        self.register_buffer("relative_position_index", relative_coords.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q * self.scale) @ k.transpose(-2, -1)

        bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        attn = attn + bias.permute(2, 0, 1).contiguous().unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.attn_drop(self.softmax(attn))
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features, drop=0.):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class SwinTransformerBlock(nn.Module):
    """W-MSA on even blocks, SW-MSA (shifted) on odd blocks -- standard
    Swin alternation so windows see across their own boundaries every
    other block instead of being permanently blind to neighboring windows."""

    def __init__(self, dim, num_heads, window_size=8, shift_size=0,
                 mlp_ratio=2., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        assert 0 <= shift_size < window_size
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, to_2tuple(window_size), num_heads,
                                     qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop=drop)

    def _attn_mask(self, x_size, device):
        H, W = x_size
        img_mask = torch.zeros((1, H, W, 1), device=device)
        slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size),
                  slice(-self.shift_size, None))
        cnt = 0
        for h in slices:
            for w in slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        mask_windows = window_partition(img_mask, self.window_size).view(-1, self.window_size ** 2)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        return attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)

    def forward(self, x, x_size):
        H, W = x_size
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x).view(B, H, W, C)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            attn_mask = self._attn_mask(x_size, x.device)
        else:
            attn_mask = None

        x_windows = window_partition(x, self.window_size).view(-1, self.window_size ** 2, C)
        attn_windows = self.attn(x_windows, mask=attn_mask).view(-1, self.window_size, self.window_size, C)
        x = window_reverse(attn_windows, self.window_size, H, W)

        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        x = x.view(B, H * W, C)

        x = shortcut + x
        return x + self.mlp(self.norm2(x))


class BasicLayer(nn.Module):
    """One RSTB's stack of Swin blocks (alternating shift)."""

    def __init__(self, dim, depth, num_heads, window_size, mlp_ratio, qkv_bias, drop, attn_drop):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim, num_heads, window_size,
                                  shift_size=0 if i % 2 == 0 else window_size // 2,
                                  mlp_ratio=mlp_ratio, qkv_bias=qkv_bias,
                                  drop=drop, attn_drop=attn_drop)
            for i in range(depth)
        ])

    def forward(self, x, x_size):
        for blk in self.blocks:
            x = blk(x, x_size)
        return x


class RSTB(nn.Module):
    """Residual Swin Transformer Block: BasicLayer + conv, with a residual
    skip around the whole thing -- the conv restores local/translation-
    equivariant inductive bias that pure window attention lacks."""

    def __init__(self, dim, depth, num_heads, window_size, mlp_ratio, qkv_bias, drop, attn_drop):
        super().__init__()
        self.residual_group = BasicLayer(dim, depth, num_heads, window_size, mlp_ratio,
                                          qkv_bias, drop, attn_drop)
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)

    def forward(self, x, x_size):
        B, L, C = x.shape
        H, W = x_size
        res = self.residual_group(x, x_size)
        res = res.transpose(1, 2).contiguous().view(B, C, H, W)
        res = self.conv(res).flatten(2).transpose(1, 2)
        return res + x


class SwinIR(nn.Module):
    def __init__(self, embed_dim=60, depths=(4, 4, 4, 4), num_heads=(6, 6, 6, 6),
                 window_size=8, mlp_ratio=2., qkv_bias=True, scale_factor=2):
        super().__init__()
        self.window_size = window_size
        self.scale_factor = scale_factor

        self.conv_first = nn.Conv2d(1, embed_dim, 3, 1, 1)

        self.layers = nn.ModuleList([
            RSTB(embed_dim, depths[i], num_heads[i], window_size, mlp_ratio, qkv_bias, 0., 0.)
            for i in range(len(depths))
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)

        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * scale_factor ** 2, 3, 1, 1),
            nn.PixelShuffle(scale_factor),
        )
        self.conv_last = nn.Conv2d(embed_dim, 1, 3, 1, 1)

        nn.init.zeros_(self.conv_last.weight)
        nn.init.zeros_(self.conv_last.bias)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    def _pad_to_window(self, x):
        """Window attention needs H, W divisible by window_size. Reflect-pad
        up to the next multiple; forward() crops the padding back off at
        the end (scaled up by scale_factor)."""
        _, _, h, w = x.shape
        m = self.window_size
        pad_h, pad_w = (m - h % m) % m, (m - w % m) % m
        return F.pad(x, (0, pad_w, 0, pad_h), mode='reflect'), pad_h, pad_w

    def forward(self, x):
        h_in, w_in = x.shape[2], x.shape[3]
        x_pad, _, _ = self._pad_to_window(x)
        x_size = (x_pad.shape[2], x_pad.shape[3])

        feat = self.conv_first(x_pad)
        deep = feat.flatten(2).transpose(1, 2)  # B,C,H,W -> B,HW,C
        for layer in self.layers:
            deep = layer(deep, x_size)
        deep = self.norm(deep).transpose(1, 2).contiguous().view(feat.shape)
        deep = self.conv_after_body(deep) + feat

        out = self.conv_last(self.upsample(deep))
        base = F.interpolate(x_pad, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        out = torch.clamp(out + base, 0.0, 1.0)

        H_out, W_out = h_in * self.scale_factor, w_in * self.scale_factor
        return out[:, :, :H_out, :W_out]

## 5. Hardware-aware training engine

`nn.DataParallel`-based, bf16 autocast on modern GPUs (fp16 + GradScaler on
older ones), with gradient accumulation whose micro-step counter persists
across epochs (a partial group at an epoch boundary correctly carries into
the next epoch instead of being dropped -- `flush()` handles the true end
of training).

In [6]:
class GradAccumState:
    """Tracks micro-batch position for gradient accumulation across the
    whole training run (not reset per epoch)."""

    def __init__(self, accum_steps):
        self.accum_steps = accum_steps
        self.micro_step = 0
        self.pending = False

    def should_zero_grad(self):
        return self.micro_step % self.accum_steps == 0

    def after_backward(self):
        self.micro_step += 1
        self.pending = True
        do_step = (self.micro_step % self.accum_steps == 0)
        if do_step:
            self.pending = False
        return do_step

    def flush_needed(self):
        return self.pending


class HardwareEngine:
    def __init__(self, model, batch_size, target_batch_size=32, device="cuda"):
        self.device = torch.device(device)
        self.gpu_name = torch.cuda.get_device_name(0)
        self.major, self.minor = torch.cuda.get_device_capability(0)
        self.num_gpus = torch.cuda.device_count()

        if self.major >= 8:
            self.precision = torch.bfloat16
            self.use_scaler = False
            print(f"--- [Modern GPU] Using bfloat16 on {self.gpu_name} ---")
        else:
            self.precision = torch.float16
            self.use_scaler = True
            print(f"--- [Legacy GPU] Using float16 + Scaler on {self.gpu_name} ---")

        self.scaler = torch.amp.GradScaler('cuda', enabled=self.use_scaler)

        self.accum_steps = max(1, target_batch_size // batch_size)
        actual_effective_batch = self.accum_steps * batch_size
        print(f"--- [Compute] DataLoader batch_size={batch_size}, "
              f"target_batch_size={target_batch_size} "
              f"-> accum_steps={self.accum_steps} "
              f"(actual effective batch = {actual_effective_batch}) ---")

        self._accum_state = GradAccumState(self.accum_steps)

        self.model = model.to(self.device)
        self._compiled = False

        if self.num_gpus > 1:
            print(f"--- [Multi-GPU] {self.num_gpus} GPUs found. Wrapping in DataParallel. ---")
            self.model = nn.DataParallel(self.model)

    def compile_model(self):
        if self._compiled:
            return self.model
        try:
            print("--- [Optimizer] Attempting torch.compile... ---")
            if isinstance(self.model, nn.DataParallel):
                inner = torch.compile(self.model.module)
                self.model = nn.DataParallel(inner)
            else:
                self.model = torch.compile(self.model)
            self._compiled = True
        except Exception as e:
            print(f"--- [Optimizer] Compile failed: {e}. Using eager mode. ---")
        return self.model

    def train_step(self, optimizer, criterion, lr_scheduler, input_img, gt_img):
        self.model.train()

        if self._accum_state.should_zero_grad():
            optimizer.zero_grad(set_to_none=True)

        input_img = input_img.to(self.device, non_blocking=True)
        gt_img = gt_img.to(self.device, non_blocking=True)

        with torch.amp.autocast('cuda', dtype=self.precision):
            output = self.model(input_img)
            loss = criterion(output, gt_img) / self.accum_steps

        if self.use_scaler:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        do_step = self._accum_state.after_backward()

        if do_step:
            if self.use_scaler:
                self.scaler.step(optimizer)
                self.scaler.update()
            else:
                optimizer.step()
            if lr_scheduler is not None and not isinstance(
                    lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step()

        return loss.item() * self.accum_steps

    def flush(self, optimizer):
        if not self._accum_state.flush_needed():
            return False
        if self.use_scaler:
            self.scaler.step(optimizer)
            self.scaler.update()
        else:
            optimizer.step()
        self._accum_state.pending = False
        print("--- [Compute] Flushed leftover accumulated gradient at end of training. ---")
        return True

    @torch.no_grad()
    def eval_step(self, criterion, input_img, gt_img):
        self.model.eval()
        input_img = input_img.to(self.device, non_blocking=True)
        gt_img = gt_img.to(self.device, non_blocking=True)

        with torch.amp.autocast('cuda', dtype=self.precision):
            output = self.model(input_img)
            loss = criterion(output, gt_img)

        return loss.item(), output

## 6. Metrics (shared by training and evaluation)

In [7]:
def compute_psnr(pred, target, max_val=1.0):
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float('inf')
    return 10 * np.log10(max_val ** 2 / mse)


def _gaussian_window(window_size=11, sigma=1.5, device="cpu"):
    coords = torch.arange(window_size, dtype=torch.float32, device=device) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g.unsqueeze(1) @ g.unsqueeze(0)
    return window_2d.unsqueeze(0).unsqueeze(0)  # (1, 1, k, k)


def compute_ssim(pred, target, window_size=11, data_range=1.0):
    """Standard single-channel SSIM. pred/target: (1, 1, H, W), same device."""
    window = _gaussian_window(window_size, device=pred.device)
    pad = window_size // 2
    C1 = (0.01 * data_range) ** 2
    C2 = (0.03 * data_range) ** 2

    mu1 = F.conv2d(pred, window, padding=pad)
    mu2 = F.conv2d(target, window, padding=pad)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2

    sigma1_sq = F.conv2d(pred * pred, window, padding=pad) - mu1_sq
    sigma2_sq = F.conv2d(target * target, window, padding=pad) - mu2_sq
    sigma12 = F.conv2d(pred * target, window, padding=pad) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean().item()

_MS_SSIM_WEIGHTS = (0.0448, 0.2856, 0.3001, 0.2363, 0.1333)  # Wang/Simoncelli/Bovik 2003


def compute_ms_ssim(pred, target, window_size=11, data_range=1.0, eps=1e-6):
    """Multi-scale SSIM metric version -- see KLAMetrologyLoss.ms_ssim for
    the loss version (same math, this one has no grad requirement)."""
    window = _gaussian_window(window_size, device=pred.device)
    pad = window_size // 2
    C1, C2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2

    cs_vals, ssim_val = [], None
    x, y = pred, target
    for i in range(len(_MS_SSIM_WEIGHTS)):
        mu1 = F.conv2d(x, window, padding=pad)
        mu2 = F.conv2d(y, window, padding=pad)
        mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
        sigma1_sq = F.conv2d(x * x, window, padding=pad) - mu1_sq
        sigma2_sq = F.conv2d(y * y, window, padding=pad) - mu2_sq
        sigma12 = F.conv2d(x * y, window, padding=pad) - mu1_mu2

        cs = ((2 * sigma12 + C2) / (sigma1_sq + sigma2_sq + C2)).mean().clamp(min=eps)
        if i == len(_MS_SSIM_WEIGHTS) - 1:
            ssim_val = (((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) /
                        ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))).mean().clamp(min=eps)
        else:
            cs_vals.append(cs)
            x, y = F.avg_pool2d(x, kernel_size=2), F.avg_pool2d(y, kernel_size=2)

    weights = torch.tensor(_MS_SSIM_WEIGHTS, device=pred.device, dtype=pred.dtype)
    cs_stack = torch.stack(cs_vals)
    result = ssim_val ** weights[-1] * torch.prod(cs_stack ** weights[:-1])
    return result.item()

## 7. Train Restormer

Mirrors `train.py`. Saves to `<checkpoint_dir>/best_model2.pt`.

In [8]:
def train_restormer(data_root, dim=64, num_blocks=3, batch_size=4, target_batch_size=32,
                     scale_factor=2, edge_weight=0.1, freq_weight=0.15, lr=2e-4,
                     weight_decay=1e-4, scheduler_patience=2, early_stop_patience=6,
                     num_workers=None, num_epochs=50, checkpoint_dir="./checkpoints"):
    num_workers = num_workers if num_workers is not None else min(8, os.cpu_count() or 1)
    config = dict(dim=dim, num_blocks=num_blocks, batch_size=batch_size,
                  target_batch_size=target_batch_size, scale_factor=scale_factor,
                  edge_weight=edge_weight, freq_weight=freq_weight, lr=lr,
                  weight_decay=weight_decay, scheduler_patience=scheduler_patience,
                  early_stop_patience=early_stop_patience, num_workers=num_workers,
                  num_epochs=num_epochs, use_compile=False, checkpoint_dir=checkpoint_dir)
    print("Config:", config)

    if not torch.cuda.is_available():
        print("WARNING: no CUDA device found -- HardwareEngine requires CUDA.")
        return

    os.makedirs(checkpoint_dir, exist_ok=True)

    train_loader, val_loader, test_loader = build_dataloaders(
        data_root, scale_factor=scale_factor, batch_size=batch_size, num_workers=num_workers)

    model = SemiRestoreNet_V2(dim=dim, num_blocks=num_blocks, scale_factor=scale_factor)
    criterion = KLAMetrologyLoss(edge_weight=edge_weight, freq_weight=freq_weight)

    engine = HardwareEngine(model, batch_size=batch_size, target_batch_size=target_batch_size)
    criterion = criterion.to(engine.device)
    optimizer = build_optimizer(engine.model, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=scheduler_patience)

    best_val_loss = float('inf')
    epochs_since_improvement = 0

    for epoch in range(num_epochs):
        t0 = time.time()
        train_losses = []
        for lr_img, gt_img in train_loader:
            loss = engine.train_step(optimizer, criterion, None, lr_img, gt_img)
            train_losses.append(loss)
        avg_train_loss = sum(train_losses) / len(train_losses)

        val_losses, val_psnrs = [], []
        for lr_img, gt_img in val_loader:
            loss, pred = engine.eval_step(criterion, lr_img, gt_img)
            val_losses.append(loss)
            val_psnrs.append(compute_psnr(pred, gt_img.to(engine.device)))
        avg_val_loss = sum(val_losses) / len(val_losses)
        avg_val_psnr = sum(val_psnrs) / len(val_psnrs)

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        dt = time.time() - t0
        print(f"Epoch {epoch+1}/{num_epochs} [{dt:.1f}s] train_loss={avg_train_loss:.4f} "
              f"val_loss={avg_val_loss:.4f} val_psnr={avg_val_psnr:.2f}dB lr={new_lr:.2e}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_since_improvement = 0
            ckpt_path = os.path.join(checkpoint_dir, "best_model2.pt")
            state_dict = (engine.model.module.state_dict()
                          if isinstance(engine.model, torch.nn.DataParallel)
                          else engine.model.state_dict())
            torch.save({"model_state": state_dict, "config": config, "epoch": epoch,
                        "val_loss": avg_val_loss, "val_psnr": avg_val_psnr, "lr": new_lr}, ckpt_path)
            print(f"  -> new best val_loss, saved to {ckpt_path}")
        else:
            epochs_since_improvement += 1
            print(f"  -> val_loss did not improve ({epochs_since_improvement}/{early_stop_patience})")
            if epochs_since_improvement >= early_stop_patience:
                if new_lr < lr:
                    print("STOPPING: model has fine-tuned at a lower LR and plateaued.")
                    break
                else:
                    print("WAITING: val_loss stalled but LR hasn't dropped yet.")

    engine.flush(optimizer)
    print("Restormer training finished.")

## 8. Train SwinIR

Mirrors `train_swinir.py`. Saves to `<checkpoint_dir>/best_model.pt`.

In [9]:
def train_swinir(data_root, embed_dim=60, depths=(4, 4, 4, 4), num_heads=(6, 6, 6, 6),
                  window_size=8, mlp_ratio=2.0, batch_size=4, target_batch_size=32,
                  scale_factor=2, edge_weight=0.5, freq_weight=0.0, ssim_weight=0.0,
                  lr=2e-4, weight_decay=1e-4, scheduler_patience=3, early_stop_patience=7,
                  num_workers=None, num_epochs=50, checkpoint_dir="./checkpoints_swinir"):
    num_workers = num_workers if num_workers is not None else min(8, os.cpu_count() or 1)
    config = dict(model_type="swinir", embed_dim=embed_dim, depths=depths, num_heads=num_heads,
                  window_size=window_size, mlp_ratio=mlp_ratio, batch_size=batch_size,
                  target_batch_size=target_batch_size, scale_factor=scale_factor,
                  edge_weight=edge_weight, freq_weight=freq_weight, ssim_weight=ssim_weight,
                  lr=lr, weight_decay=weight_decay, scheduler_patience=scheduler_patience,
                  early_stop_patience=early_stop_patience, num_workers=num_workers,
                  num_epochs=num_epochs, use_compile=False, checkpoint_dir=checkpoint_dir)
    print("Config:", config)

    if not torch.cuda.is_available():
        print("WARNING: no CUDA device found -- HardwareEngine requires CUDA.")
        return

    os.makedirs(checkpoint_dir, exist_ok=True)

    train_loader, val_loader, test_loader = build_dataloaders(
        data_root, scale_factor=scale_factor, batch_size=batch_size, num_workers=num_workers)

    model = SwinIR(embed_dim=embed_dim, depths=depths, num_heads=num_heads,
                    window_size=window_size, mlp_ratio=mlp_ratio, scale_factor=scale_factor)
    criterion = KLAMetrologyLoss(edge_weight=edge_weight, freq_weight=freq_weight, ssim_weight=ssim_weight)

    engine = HardwareEngine(model, batch_size=batch_size, target_batch_size=target_batch_size)
    criterion = criterion.to(engine.device)
    optimizer = build_optimizer(engine.model, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=scheduler_patience)

    best_val_loss = float('inf')
    epochs_since_improvement = 0

    for epoch in range(num_epochs):
        t0 = time.time()
        train_losses = []
        for lr_img, gt_img in train_loader:
            loss = engine.train_step(optimizer, criterion, None, lr_img, gt_img)
            train_losses.append(loss)
        avg_train_loss = sum(train_losses) / len(train_losses)

        val_losses, val_psnrs = [], []
        for lr_img, gt_img in val_loader:
            loss, pred = engine.eval_step(criterion, lr_img, gt_img)
            val_losses.append(loss)
            val_psnrs.append(compute_psnr(pred, gt_img.to(engine.device)))
        avg_val_loss = sum(val_losses) / len(val_losses)
        avg_val_psnr = sum(val_psnrs) / len(val_psnrs)

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        dt = time.time() - t0
        print(f"Epoch {epoch+1}/{num_epochs} [{dt:.1f}s] train_loss={avg_train_loss:.4f} "
              f"val_loss={avg_val_loss:.4f} val_psnr={avg_val_psnr:.2f}dB lr={new_lr:.2e}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_since_improvement = 0
            ckpt_path = os.path.join(checkpoint_dir, "best_model.pt")
            state_dict = (engine.model.module.state_dict()
                          if isinstance(engine.model, torch.nn.DataParallel)
                          else engine.model.state_dict())
            torch.save({"model_state": state_dict, "config": config, "epoch": epoch,
                        "val_loss": avg_val_loss, "val_psnr": avg_val_psnr, "lr": new_lr}, ckpt_path)
            print(f"  -> new best val_loss, saved to {ckpt_path}")
        else:
            epochs_since_improvement += 1
            print(f"  -> val_loss did not improve ({epochs_since_improvement}/{early_stop_patience})")
            if epochs_since_improvement >= early_stop_patience:
                if new_lr < lr:
                    print("STOPPING: model has fine-tuned at a lower LR and plateaued.")
                    break
                else:
                    print("WAITING: val_loss stalled but LR hasn't dropped yet.")

    engine.flush(optimizer)
    print("SwinIR training finished.")

## 8b. Fine-tune from a checkpoint (loss-ablation experiments)

Generic resume-and-fine-tune: loads an existing checkpoint's weights,
swaps in a new loss configuration (e.g. adding `ssim_weight` or
`ms_ssim_weight`), and continues training at a lower LR. This is how
the SSIM and MS-SSIM loss experiments documented in `swinirapproach.md`
were actually run -- reuses the same held-out val split (fixed seed=42)
so the comparison sample stays genuinely unseen throughout.

In [10]:
def finetune_restormer(data_root, base_ckpt_path, out_ckpt_path, loss_kwargs,
                       lr=5e-5, scheduler_patience=2, early_stop_patience=5,
                       num_epochs=15, num_workers=None):
    """loss_kwargs overrides KLAMetrologyLoss kwargs on top of the base
    checkpoint's own (e.g. {"ms_ssim_weight": 0.15})."""
    num_workers = num_workers if num_workers is not None else min(8, os.cpu_count() or 1)
    device = "cuda"
    base_ckpt = torch.load(base_ckpt_path, map_location=device)
    base_cfg = base_ckpt["config"]
    print("Base checkpoint config:", base_cfg)
    print(f"Base checkpoint: epoch={base_ckpt['epoch']+1} val_loss={base_ckpt['val_loss']:.4f} "
          f"val_psnr={base_ckpt['val_psnr']:.2f}dB")

    config = dict(base_cfg)
    config.update(lr=lr, scheduler_patience=scheduler_patience,
                  early_stop_patience=early_stop_patience, num_epochs=num_epochs,
                  num_workers=num_workers, finetune_from=base_ckpt_path, **loss_kwargs)
    print("Fine-tune config:", config)

    train_loader, val_loader, test_loader = build_dataloaders(
        data_root, scale_factor=config["scale_factor"],
        batch_size=config["batch_size"], num_workers=num_workers)

    model = SemiRestoreNet_V2(dim=config["dim"], num_blocks=config["num_blocks"],
                               scale_factor=config["scale_factor"])
    model.load_state_dict(base_ckpt["model_state"])
    print("Loaded base weights into fresh model.")

    loss_arg_names = {"edge_weight", "freq_weight", "ssim_weight", "ms_ssim_weight"}
    criterion = KLAMetrologyLoss(**{k: v for k, v in config.items() if k in loss_arg_names})

    engine = HardwareEngine(model, batch_size=config["batch_size"],
                             target_batch_size=config["target_batch_size"])
    criterion = criterion.to(engine.device)
    optimizer = build_optimizer(engine.model, lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=config["scheduler_patience"])

    best_val_loss = float('inf')
    epochs_since_improvement = 0

    for epoch in range(config["num_epochs"]):
        t0 = time.time()
        train_losses = []
        for lr_img, gt_img in train_loader:
            loss = engine.train_step(optimizer, criterion, None, lr_img, gt_img)
            train_losses.append(loss)
        avg_train_loss = sum(train_losses) / len(train_losses)

        val_losses, val_psnrs = [], []
        for lr_img, gt_img in val_loader:
            loss, pred = engine.eval_step(criterion, lr_img, gt_img)
            val_losses.append(loss)
            val_psnrs.append(compute_psnr(pred, gt_img.to(engine.device)))
        avg_val_loss = sum(val_losses) / len(val_losses)
        avg_val_psnr = sum(val_psnrs) / len(val_psnrs)

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(avg_val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        dt = time.time() - t0
        print(f"Epoch {epoch+1}/{config['num_epochs']} [{dt:.1f}s] "
              f"train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} "
              f"val_psnr={avg_val_psnr:.2f}dB lr={new_lr:.2e}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_since_improvement = 0
            state_dict = (engine.model.module.state_dict()
                          if isinstance(engine.model, torch.nn.DataParallel)
                          else engine.model.state_dict())
            torch.save({"model_state": state_dict, "config": config, "epoch": epoch,
                        "val_loss": avg_val_loss, "val_psnr": avg_val_psnr, "lr": new_lr,
                        "calib_p_low": base_ckpt.get("calib_p_low"),
                        "calib_p_high": base_ckpt.get("calib_p_high")}, out_ckpt_path)
            print(f"  -> new best val_loss, saved to {out_ckpt_path}")
        else:
            epochs_since_improvement += 1
            print(f"  -> no improve ({epochs_since_improvement}/{config['early_stop_patience']})")
            if epochs_since_improvement >= config["early_stop_patience"] and new_lr < config["lr"]:
                print("STOPPING early.")
                break

    engine.flush(optimizer)
    print("Fine-tune finished.")
    return out_ckpt_path

## 9. Evaluation

Mirrors `evaluation.py`. `build_model` branches on the checkpoint's
`model_type` (absent means Restormer, for checkpoints saved before the
SwinIR branch existed), so the same `evaluate()` works for either.

In [11]:
def build_model(cfg, device):
    model_type = cfg.get("model_type", "restormer")
    if model_type == "swinir":
        return SwinIR(
            embed_dim=cfg["embed_dim"], depths=cfg["depths"], num_heads=cfg["num_heads"],
            window_size=cfg["window_size"], mlp_ratio=cfg["mlp_ratio"],
            scale_factor=cfg["scale_factor"],
        ).to(device)
    return SemiRestoreNet_V2(
        dim=cfg["dim"], num_blocks=cfg["num_blocks"], scale_factor=cfg["scale_factor"],
    ).to(device)


def save_comparison_plot(lr_np, pred_np, gt_np, title, out_path):
    n_panels = 3 if gt_np is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4))

    axes[0].imshow(lr_np, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title("Input (noisy LR)")
    axes[0].axis('off')

    axes[1].imshow(pred_np, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Model prediction")
    axes[1].axis('off')

    if gt_np is not None:
        axes[2].imshow(gt_np, cmap='gray', vmin=0, vmax=1)
        axes[2].set_title("Ground truth")
        axes[2].axis('off')

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close(fig)


def evaluate(data_root, ckpt_path, out_dir="./eval_outputs"):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    ckpt = torch.load(ckpt_path, map_location=device)
    cfg = ckpt["config"]
    print(f"Loaded checkpoint from epoch {ckpt['epoch']+1}, "
          f"val_loss={ckpt['val_loss']:.4f}, val_psnr={ckpt['val_psnr']:.2f}dB "
          f"(as recorded during training)")
    print(f"Checkpoint config: {cfg}")

    model = build_model(cfg, device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    p_low, p_high = load_calib_stats(data_root)

    full_train = SEMPairDataset(
        gt_dir=os.path.join(data_root, "train", "gt"),
        lr_dir=os.path.join(data_root, "train", "NoisyLR"),
        p_low=p_low, p_high=p_high,
        scale_factor=cfg["scale_factor"], augment=False,
    )
    val_fraction = 0.1
    seed = 42
    n_val = max(1, int(len(full_train) * val_fraction))
    n_train = len(full_train) - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = torch.utils.data.random_split(
        full_train, [n_train, n_val], generator=generator
    )

    print(f"\nEvaluating on {len(val_subset)} held-out validation samples...")
    results = []
    with torch.no_grad():
        for idx in val_subset.indices:
            lr_img, gt_img = full_train[idx]
            lr_img = lr_img.unsqueeze(0).to(device)
            gt_img = gt_img.unsqueeze(0).to(device)
            pred = model(lr_img)
            psnr = compute_psnr(pred, gt_img)
            ssim = compute_ssim(pred, gt_img)
            fname = os.path.basename(full_train.pairs[idx][0])
            results.append((psnr, ssim, fname))

    psnrs = np.array([r[0] for r in results])
    ssims = np.array([r[1] for r in results])
    print(f"\n=== Validation set summary (n={len(results)}) ===")
    print(f"PSNR: mean={psnrs.mean():.2f}dB  std={psnrs.std():.2f}  "
          f"min={psnrs.min():.2f}  max={psnrs.max():.2f}")
    print(f"SSIM: mean={ssims.mean():.4f}  std={ssims.std():.4f}  "
          f"min={ssims.min():.4f}  max={ssims.max():.4f}")

    results_sorted = sorted(results, key=lambda r: r[0])
    print(f"\nWorst 5 samples by PSNR:")
    for psnr, ssim, fname in results_sorted[:5]:
        print(f"  {fname}: PSNR={psnr:.2f}dB, SSIM={ssim:.4f}")
    print(f"\nBest 5 samples by PSNR:")
    for psnr, ssim, fname in results_sorted[-5:]:
        print(f"  {fname}: PSNR={psnr:.2f}dB, SSIM={ssim:.4f}")

    name_to_idx = {os.path.basename(full_train.pairs[i][0]): i for i in val_subset.indices}
    for psnr, ssim, fname in results_sorted[:3]:
        idx = name_to_idx[fname]
        lr_img, gt_img = full_train[idx]
        with torch.no_grad():
            pred = model(lr_img.unsqueeze(0).to(device)).squeeze(0).cpu()
        lr_up = F.interpolate(lr_img.unsqueeze(0), scale_factor=cfg["scale_factor"],
                               mode='nearest').squeeze(0)
        save_comparison_plot(
            lr_up[0].numpy(), pred[0].numpy(), gt_img[0].numpy(),
            title=f"{fname} (worst case: PSNR={psnr:.2f}dB, SSIM={ssim:.4f})",
            out_path=os.path.join(out_dir, f"worst0_{fname.replace('.npy','')}.png"),
        )
    print(f"\nSaved worst-3 comparison plots to {out_dir}/")

    print(f"\n=== Test set (no ground truth -- visuals only, no PSNR/SSIM) ===")
    test_set = SEMTestDataset(os.path.join(data_root, "test"), p_low, p_high)
    n_show = min(5, len(test_set))
    for i in range(n_show):
        lr_img, fname = test_set[i]
        with torch.no_grad():
            pred = model(lr_img.unsqueeze(0).to(device)).squeeze(0).cpu()
        lr_up = F.interpolate(lr_img.unsqueeze(0), scale_factor=cfg["scale_factor"],
                               mode='nearest').squeeze(0)
        save_comparison_plot(
            lr_up[0].numpy(), pred[0].numpy(), None,
            title=f"TEST (no GT): {fname}",
            out_path=os.path.join(out_dir, f"test_{fname.replace('.npy','')}.png"),
        )
    print(f"Saved {n_show} test-set prediction visuals to {out_dir}/")
    print(f"\nDone. All outputs in {out_dir}/")

    return {"psnr_mean": psnrs.mean(), "psnr_std": psnrs.std(),
            "ssim_mean": ssims.mean(), "ssim_std": ssims.std(), "results": results}

## 10. Example usage

Uncomment what you need. Full 50-epoch runs take ~2h (Restormer) /
~3h (SwinIR) on an 8GB laptop GPU -- see `swinirapproach.md` for the
actual numbers this pipeline already produced.

In [12]:
DATA_ROOT = "data"

# run_calibration(DATA_ROOT)

# train_restormer(DATA_ROOT, num_epochs=50)
# train_swinir(DATA_ROOT, num_epochs=50)

# evaluate(DATA_ROOT, "checkpoints/best_model2.pt")
# evaluate(DATA_ROOT, "checkpoints_swinir/best_model.pt")

# Loss-ablation experiments (see swinirapproach.md for results):
# finetune_restormer(DATA_ROOT, "checkpoints/best_model2.pt",
#                     "checkpoints/best_model_ssim.pt", {"ssim_weight": 0.3})
# finetune_restormer(DATA_ROOT, "checkpoints/best_model2.pt",
#                     "checkpoints/best_model_msssim.pt", {"ms_ssim_weight": 0.15})

## 11. MS-SSIM loss experiment -- actual run

Ran via:
```python
finetune_restormer(DATA_ROOT, "checkpoints/best_model2.pt",
                    "checkpoints/best_model_msssim.pt", {"ms_ssim_weight": 0.15})
```

Not re-executed in this pass -- it already ran for real (~31 minutes) and
produced `checkpoints/best_model_msssim.pt`; re-running would cost another
~31 minutes of GPU time for a result that already exists on disk (and
training isn't exactly reproducible run-to-run anyway, due to shuffling
and augmentation). The log below is the actual captured output from that
run, reproduced verbatim.

**Result: training collapsed at epoch 4** (val_psnr 20.95dB -> 7.73dB,
never recovered, early-stopped at epoch 8). The best checkpoint that was
actually saved is from epoch 3 (val_psnr=20.95dB) -- still a large
regression from the 27.91dB baseline, but not the catastrophic ~7.7dB the
raw end-of-training model state would suggest. Live evaluation of that
actual saved checkpoint is in the next cell.

In [13]:
# Real captured log from the actual run (see markdown above) --
# reproduced here as output, not re-executed.
print(open("C:/Users/LENOVO/AppData/Local/Temp/claude/c--Users-LENOVO-OneDrive-Desktop-kla-hackathon/bdaadd7a-df15-4748-8b69-efebaf3ba17b/scratchpad/finetune_msssim.log").read().split("=== Before/after")[0])

Base checkpoint config: {'dim': 64, 'num_blocks': 3, 'batch_size': 4, 'target_batch_size': 32, 'scale_factor': 2, 'edge_weight': 0.1, 'freq_weight': 0.15, 'lr': 0.0002, 'weight_decay': 0.0001, 'scheduler_patience': 2, 'early_stop_patience': 6, 'num_workers': 8, 'num_epochs': 50, 'use_compile': False, 'checkpoint_dir': './checkpoints'}
Base checkpoint: epoch=50 val_loss=0.0394 val_psnr=27.91dB
Fine-tune config: {'dim': 64, 'num_blocks': 3, 'batch_size': 4, 'target_batch_size': 32, 'scale_factor': 2, 'edge_weight': 0.1, 'freq_weight': 0.15, 'lr': 5e-05, 'weight_decay': 0.0001, 'scheduler_patience': 2, 'early_stop_patience': 5, 'num_workers': 8, 'num_epochs': 15, 'use_compile': False, 'checkpoint_dir': './checkpoints', 'ms_ssim_weight': 0.15, 'finetune_phase': 'ms_ssim', 'finetune_from': 'checkpoints/best_model2.pt'}
[SEMPairDataset] 3200 paired samples found.
[SEMTestDataset] 400 test samples found.
Loaded base weights into fresh model.
--- [Modern GPU] Using bfloat16 on NVIDIA GeForce R

In [14]:
# Genuinely live-executed right now (GPU free, background job finished).
# Corrects the original scratch script's bug: that one compared against
# the LIVE end-of-training model object (the collapsed epoch-8 state),
# not the actual best-saved checkpoint (epoch 3) -- this reloads from
# disk, so it's checking exactly what got saved.
print("=== Live evaluation: checkpoints/best_model_msssim.pt (actual saved checkpoint) ===")
metrics_msssim = evaluate(DATA_ROOT, "checkpoints/best_model_msssim.pt")

print("\n=== Corrected single-sample check: 002982.npy, reloaded from the saved checkpoint ===")
ckpt_before = torch.load("checkpoints/best_model2.pt", map_location=DEVICE)
ckpt_after = torch.load("checkpoints/best_model_msssim.pt", map_location=DEVICE)

model_before = build_model(ckpt_before["config"], DEVICE)
model_before.load_state_dict(ckpt_before["model_state"])
model_before.eval()

model_after = build_model(ckpt_after["config"], DEVICE)
model_after.load_state_dict(ckpt_after["model_state"])
model_after.eval()

p_low, p_high = load_calib_stats(DATA_ROOT)
full_train = SEMPairDataset(
    gt_dir="data/train/gt", lr_dir="data/train/NoisyLR",
    p_low=p_low, p_high=p_high, scale_factor=2, augment=False,
)
name_to_idx = {os.path.basename(p[0]): i for i, p in enumerate(full_train.pairs)}
idx = name_to_idx["002982.npy"]
lr_img, gt_img = full_train[idx]
lr_b = lr_img.unsqueeze(0).to(DEVICE)
gt_b = gt_img.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred_before = model_before(lr_b)
    pred_after = model_after(lr_b)

psnr_before = compute_psnr(pred_before, gt_b)
ssim_before = compute_ssim(pred_before, gt_b)
msssim_before = compute_ms_ssim(pred_before, gt_b)
psnr_after = compute_psnr(pred_after, gt_b)
ssim_after = compute_ssim(pred_after, gt_b)
msssim_after = compute_ms_ssim(pred_after, gt_b)

print(f"BEFORE (baseline, 50 epochs):        PSNR={psnr_before:.2f}dB  SSIM={ssim_before:.4f}  MS-SSIM={msssim_before:.4f}")
print(f"AFTER  (ms_ssim_weight=0.15, ep. 3):  PSNR={psnr_after:.2f}dB  SSIM={ssim_after:.4f}  MS-SSIM={msssim_after:.4f}")
print(f"Delta: PSNR {psnr_after-psnr_before:+.2f}dB, SSIM {ssim_after-ssim_before:+.4f}, MS-SSIM {msssim_after-msssim_before:+.4f}")


=== Live evaluation: checkpoints/best_model_msssim.pt (actual saved checkpoint) ===
Using device: cuda
Loaded checkpoint from epoch 3, val_loss=0.1247, val_psnr=20.95dB (as recorded during training)
Checkpoint config: {'dim': 64, 'num_blocks': 3, 'batch_size': 4, 'target_batch_size': 32, 'scale_factor': 2, 'edge_weight': 0.1, 'freq_weight': 0.15, 'lr': 5e-05, 'weight_decay': 0.0001, 'scheduler_patience': 2, 'early_stop_patience': 5, 'num_workers': 8, 'num_epochs': 15, 'use_compile': False, 'checkpoint_dir': './checkpoints', 'ms_ssim_weight': 0.15, 'finetune_phase': 'ms_ssim', 'finetune_from': 'checkpoints/best_model2.pt'}
[SEMPairDataset] 3200 paired samples found.

Evaluating on 320 held-out validation samples...

=== Validation set summary (n=320) ===
PSNR: mean=23.02dB  std=5.49  min=12.47  max=42.16
SSIM: mean=0.5441  std=0.2400  min=-0.0398  max=0.9421

Worst 5 samples by PSNR:
  000397.npy: PSNR=12.47dB, SSIM=0.0430
  002982.npy: PSNR=12.48dB, SSIM=0.2870
  002471.npy: PSNR=12.64